# Importation Librairie

In [1]:
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, mean_absolute_error
import numpy as np
import math
import pandas as pd

# Régression type

<b>Ajuster le modèle pour minimiser l’erreur entre prédiction et réalité (fonction de perte).

## Régression linéaire

<b>But :</b> prédire une variable quantitative continue.

<b>Principe :</b> ajuster les coefficients pour minimiser l’erreur quadratique moyenne (MSE).

<b>Utilisation typique :</b> prédiction de prix, croissance, valeurs numériques continues.

In [ ]:
# --- Régression linéaire ---
def regression_lineaire(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return model, mse, y_pred

## Régression logistique

<b>But :</b> prédire une variable qualitative (binaire ou multi-classes).

<b>Principe :</b> au lieu de prédire directement une valeur numérique, elle prédit une probabilité via la fonction sigmoïde. La classe est choisie en appliquant un seuil (souvent 0.5).

<b>Utilisation typique :</b> classification (spam / non spam, malade / sain).

In [ ]:
# --- Régression logistique ---
def regression_logistique(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return model, acc, y_pred


## Régression Ridge

<b>But :</b> régression linéaire avec régularisation L2.

<b>Principe :</b> au lieu de prédire directement une valeur numérique, elle prédit une probabilité via la fonction sigmoïde. La classe est choisie en appliquant un seuil (souvent 0.5).

<b>Utilisation typique :</b> classification (spam / non spam, malade / sain).

In [ ]:
# --- Régression Ridge ---
def regression_ridge(X, y, alpha=1.0, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return model, mse, y_pred

## Régression Lasso

<b>But :</b> régression linéaire avec régularisation L1.

<b>Principe :</b> pousse certains coefficients exactement à zéro → fait de la sélection de variables.

<b>Utilisation typique :</b> quand on veut identifier les variables les plus pertinentes.

In [ ]:
# --- Régression Lasso ---
def regression_lasso(X, y, alpha=0.1, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = Lasso(alpha=alpha)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return model, mse, y_pred

## Régression Elastic Net

<b>But :</b> combinaison de Ridge et Lasso.

<b>Principe :</b> équilibre entre la réduction de variance (Ridge) et la sélection de variables (Lasso).

<b>Utilisation typique :</b> grand nombre de variables, certaines corrélées, certaines non.

In [ ]:
# --- Régression Elastic Net ---
def regression_elasticnet(X, y, alpha=1.0, l1_ratio=0.5, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return model, mse, y_pred

# Regression --- expérimental

- <b>Premier Test :</b> Faire le carré du loss pour qu'il est plus d'impact et potentiellement permette de mieux corriger le reste du réseau.
- <b>Deuxième Test :</b> Modification des paramètres de pondération ridge lasso et taux d'apprentissage ainsi que le paramètres de régularisation.

In [2]:
class ElasticNetCustom:
    def __init__(self, alpha=1.0, l1_ratio=0.7, lr=0.01, n_iter=1000, custom_loss=None):
        """
        alpha : paramètre de régularisation global
        l1_ratio : pondération entre L1 et L2 (0 = Ridge, 1 = Lasso)
        lr : taux d'apprentissage
        n_iter : nombre d'itérations
        custom_loss : fonction de coût personnalisée f(y, y_pred, w)
        """
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.lr = lr
        self.n_iter = n_iter
        self.custom_loss = custom_loss
        self.w = None
        self.b = None

    def fit(self, X, y):
        n, p = X.shape
        self.w = np.zeros(p)
        self.b = 0

        for _ in range(self.n_iter):
            # prédictions
            y_pred = X.dot(self.w) + self.b

            # gradients standard (MSE)
            error = y_pred - y
            grad_w = (3/n) * (X.T.dot(error))
            grad_b = (3/n) * np.sum(error)

            # ajout de la régularisation Elastic Net
            grad_w += self.alpha * (
                self.l1_ratio * np.sign(self.w) + (1 - self.l1_ratio) * self.w
            )

            # mise à jour
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

    def predict(self, X):
        return X.dot(self.w) + self.b

    def loss(self, X, y):
        y_pred = self.predict(X)
        if self.custom_loss:
            # utilise la fonction de coût fournie par l'utilisateur
            return self.custom_loss(y, y_pred, self.w)
        else:
            # coût par défaut : MSE + pénalisation Elastic Net
            mse = np.mean((y - y_pred) ** 2)
            penalty = self.alpha * (
                self.l1_ratio * np.sum(np.abs(self.w)) +
                (1 - self.l1_ratio) * 0.5 * np.sum(self.w ** 2)
            )
            return mse + penalty


In [7]:
# Données jouets
np.random.seed(42)
X = np.random.randn(100, 3)
true_w = np.array([1.5, -2.0, 0.5])
y = X.dot(true_w) + np.random.randn(100) * 0.5

# Fonction de coût personnalisée (ex: MAE + pénalisation Elastic Net)
# def custom_mae(y, y_pred, w):
#     mae = np.mean(np.abs(y - y_pred))
#     penalty = 0.1 * (0.5*np.sum(np.abs(w)) + 0.5*np.sum(w**2))
#     return mae + penalty

# Modèle
model = ElasticNetCustom(alpha=0.018, l1_ratio=0.7, lr=0.01, n_iter=1000)
model.fit(X, y)

print("Poids appris :", model.w)
print("Biais :", model.b)
print("Loss (custom) :", model.loss(X, y))


Poids appris : [ 1.45078586 -2.01774775  0.44198811]
Biais : 0.05902481154044897
Loss (custom) : 0.2558147145614379


# Test et comparaison

In [9]:
# --- Fonction pour comparer ---
def compare_models(X, y, custom_model):
    # Modèles classiques
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=0.1),
        "Lasso": Lasso(alpha=0.1),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5)
    }
    
    results = []

    # Fit modèles classiques
    for name, model in models.items():
        model.fit(X, y)
        y_pred = model.predict(X)
        mse = mean_squared_error(y, y_pred)
        mae = mean_absolute_error(y, y_pred)
        results.append({"Model": name, "MSE": mse, "MAE": mae})

    # Fit modèle custom
    y_pred_custom = custom_model.predict(X)
    mse_custom = mean_squared_error(y, y_pred_custom)
    mae_custom = mean_absolute_error(y, y_pred_custom)
    results.append({"Model": "ElasticNetCustom", "MSE": mse_custom, "MAE": mae_custom})

    return pd.DataFrame(results)

In [10]:
df_results = compare_models(X, y, model)
print(df_results)

              Model       MSE       MAE
0  LinearRegression  0.189191  0.348601
1             Ridge  0.189198  0.348661
2             Lasso  0.222241  0.368111
3        ElasticNet  0.232570  0.370306
4  ElasticNetCustom  0.189339  0.348749
